In [3]:
#!/usr/bin/env python3
"""Average survival rate per (role, strategy) combo.

Survival rate for a row = alive / total for that role+strategy.
Rows where total == 0 (role/strategy not present in that game) are skipped.

Set CSV_PATH below, then run the file.
"""
import csv
from collections import defaultdict

CSV_PATH = "gamesFinale.csv"  # <-- set this to your CSV file

STRATEGIES = ["scasp", "behavior_tree", "llm"]
ROLES = ["innocents", "half_zombies"]


def main(path):
    rates = defaultdict(list)  # (role, strategy) -> [per-game survival rates]
    pooled = defaultdict(lambda: [0, 0])  # (role, strategy) -> [alive, total]

    with open(path, newline="") as f:
        for row in csv.DictReader(f):
            for strat in STRATEGIES:
                for role in ROLES:
                    total = float(row.get(f"{strat}_{role}_total") or 0)
                    alive = float(row.get(f"{strat}_{role}_alive") or 0)
                    if total == 0:
                        continue  # doesn't apply
                    rates[(role, strat)].append(alive / total)
                    pooled[(role, strat)][0] += alive
                    pooled[(role, strat)][1] += total

    print(f"{'role':<14}{'strategy':<16}{'games':>6}{'avg survival':>14}{'pooled':>10}")
    print("-" * 60)
    for role in ROLES:
        for strat in STRATEGIES:
            key = (role, strat)
            if key not in rates:
                continue
            r = rates[key]
            a, t = pooled[key]
            print(f"{role:<14}{strat:<16}{len(r):>6}{sum(r) / len(r):>13.1%}{a / t:>10.1%}")


if __name__ == "__main__":
    main(CSV_PATH)

role          strategy         games  avg survival    pooled
------------------------------------------------------------
innocents     scasp              200        63.9%     63.8%
innocents     behavior_tree      200        73.6%     73.6%
innocents     llm                200        53.5%     53.5%
half_zombies  scasp              148        38.1%     17.3%
half_zombies  behavior_tree       92        36.8%     14.2%
half_zombies  llm                 85        44.0%     25.0%
